# VIS model 

In [1]:

from utils import get_device
from data_process_v4 import get_loaders, class_cols
from data_process_v4 import TASK_3_TEST_LABELS_DIR

from VIS_V1 import create_model, train_model
from utils import get_device
import torch
import pandas as pd
from tqdm import tqdm

Using CSV_PATH: data/Task_3/ISIC2018_Task3_Training_GroundTruth.csv
Using IMAGES_DIR: data/Task_3/Train_images
Using VAL CSV: data/Task_3/ISIC2018_Task3_Validation_GroundTruth.csv
Using VAL IMAGES_DIR: data/Task_3/Validation_images
Using TEST CSV: data/Task_3/ISIC2018_Task3_Test_GroundTruth.csv
Using TEST IMAGES_DIR: data/Task_3/Test_images


In [8]:
device = get_device()

loaders = get_loaders(
    image_size=(384, 384),
    num_workers=0,
    test_csv_path=TASK_3_TEST_LABELS_DIR,
)

train_loader, val_loader, test_loader, train_df, val_df, test_df = loaders
class_cols_order = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]
num_classes = 7

class_counts = [int(train_df[c].sum()) for c in class_cols_order]
model_name = "best_model_vis.pth"
base_model = create_model(num_classes=num_classes, model_name="vit_base_patch16_384")
ckpt = torch.load(model_name, map_location=device)
base_model.load_state_dict(ckpt['model_state'])

<All keys matched successfully>

In [ ]:
model, history2 = train_model(
    train_loader,
    val_loader,
    num_classes=num_classes,
    device=device,
    epochs=3,
    lr = 3e-5,                
    class_counts=class_counts,
    model=base_model,          
    save_path=model_name
)

In [ ]:
#torch.save({"model_state": model.state_dict()}, 'best_model_vis.pth')

In [9]:
import torch
from eval_vis import evaluate_model

# device (can also reuse what you used in train_model)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_model.to(device)


metrics, cm, (y_true, y_pred) = evaluate_model(
    model=base_model,
    data_loader=test_loader,
    device=device,
    class_names=class_cols_order,   # or None
    num_classes=len(class_cols_order),  # or None to infer
)

print("Metrics dict:", metrics)


Evaluating:   0%|          | 0/95 [00:00<?, ?it/s]

Loss: 1.8151 | Accuracy: 0.4074 | Mean (macro) recall: 0.5001


Metrics dict: {'loss': 1.8151226882581357, 'accuracy': 0.4074074074074074, 'mean_recall': 0.5000634639334002}


# VIS model V3


In [ ]:
import os
import torch
from data_process_v5 import get_loaders, class_cols        
from VIS_V3 import create_model, train_model, evaluate
from utils import get_device


train_loader, val_loader, test_loader,train_df,val_df,test_df, class_weights,  num_classes, = get_loaders(
    image_size=(384, 384),
    num_workers=0,

    )

print("Train size:", len(train_df))
print("Val size:  ", len(val_df))
print("Test size: ", len(test_df))
print("Num classes:", num_classes)

class_counts = [int(train_df[c].sum()) for c in class_cols_order]
print("Class counts in train:", dict(zip(class_cols_order, class_counts)))

In [ ]:
model_name = "vit_base_patch16_384"
ckpt_path = "vis_v5.pth"

device = get_device()
model = create_model(num_classes=num_classes, model_name=model_name, pretrained=False)
model.to(device)
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state"])
print(f"Loaded existing checkpoint from {ckpt_path}")

model, history = train_model(
    train_loader=train_loader,
    val_loader=val_loader,
    num_classes=num_classes,
    model_name=model_name,
    epochs=10,              
    lr=3e-5,                
    weight_decay=0.05,
    device=None,            
    class_counts=class_counts,  
    save_path=ckpt_path,
    model=model,            
)

In [ ]:
device = get_device()

model.to(device)

print("\nValidation performance:")
val_loss, val_acc = evaluate(
    model=model,
    loader=val_loader,
    device=device,
    compute_metrics=True,          # prints per-class + macro recall
    class_names=class_cols_order,
)

print("\nTest performance:")
test_loss, test_acc = evaluate(
    model=model,
    loader=test_loader,
    device=device,
    compute_metrics=True,          # prints per-class + macro recall
    class_names=class_cols_order,
)

print(f"\nFinal: val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")
print(f"       test_loss={test_loss:.4f}, test_acc={test_acc:.4f}")

# resnet

In [7]:
import torch
from resnet_v2 import train_model, create_resnet_model
from eval_resnet import evaluate_model
from data_process_v4 import get_loaders, class_cols
from data_process_v4 import TASK_3_TEST_LABELS_DIR
from utils import get_device

In [ ]:
device = get_device()
loaders = get_loaders(
    image_size=(650, 400),                 
    num_workers=0,
    test_csv_path=TASK_3_TEST_LABELS_DIR,
)

train_loader, val_loader, test_loader, train_df, val_df, test_df = loaders

class_cols_order = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]
num_classes = 7

# class distribution for BalancedFocalLoss
class_counts = [int(train_df[c].sum()) for c in class_cols_order]

model = create_resnet_model(
     num_classes=num_classes,
     pretrained=True,
     backbone_name="resnet50",
     head_hidden_dim=512,
     head_dropout=0.3,
 )

In [ ]:


model, history = train_model(
    train_loader=train_loader,
    val_loader=val_loader,
    num_classes=num_classes,
    epochs=40,
    lr=3e-4,
    weight_decay=0.05,
    class_counts=class_counts,              
    save_path="best_resnet50_resizer.pth",
    device=device,                          
    backbone_name="resnet50",
    head_hidden_dim=512,
    head_dropout=0.3,
)

In [12]:
device = get_device()
base_model  = create_resnet_model(
     num_classes=7,
     pretrained=True,
     backbone_name="resnet50",
     head_hidden_dim=512,
     head_dropout=0.3,
 )
ckpt = torch.load("best_resnet50_resizer.pth", map_location=device)
base_model.load_state_dict(ckpt["model_state"])

<All keys matched successfully>

In [13]:
from eval_resnet import evaluate_model

device = get_device()
print("Using device:", device)

class_cols_order = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]

# evaluate on test set, compute mean recall and plot confusion matrix
mean_recall, cm, (y_true, y_pred) = evaluate_model(
    model=base_model,
    data_loader=test_loader,
    device=device,
    class_names=class_cols_order,   
    num_classes=len(class_cols_order),
    verbose=True,
)

print("Mean recall (macro):", mean_recall)

Using device: mps


Mean (macro) recall: 0.6755


Mean recall (macro): 0.6754721858151264


# Stack of the two models

In [1]:
import torch
import numpy as np
from VIS_V1 import create_model as create_vis_model
from resnet_v2 import create_resnet_model
from utils import get_device
device = get_device()

In [2]:
class_cols_order = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]
num_classes = 7

vis_model_name = "best_model_vis.pth"
best_vis_model = create_vis_model(num_classes=num_classes, model_name="vit_base_patch16_384")
ckpt = torch.load(vis_model_name, map_location=device)
best_vis_model.load_state_dict(ckpt['model_state'])

resnet_model_name = "best_resnet50_resizer.pth"
best_resnet_model = create_resnet_model( num_classes=num_classes, pretrained=True, backbone_name="resnet50", head_hidden_dim=512, head_dropout=0.3)
ckpt = torch.load(resnet_model_name, map_location=device)
best_resnet_model.load_state_dict(ckpt["model_state"])


<All keys matched successfully>

In [ ]:
best_resnet_model.to(device).eval()
best_vis_model.to(device).eval()

@torch.no_grad()
def get_probs_and_labels(model, data_loader, device):
    all_probs = []
    all_labels = []
    
    for xb, yb in data_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb)
        probs = torch.softmax(logits, dim=1)

        all_probs.append(probs.cpu())
        all_labels.append(yb.cpu())

    all_probs = torch.cat(all_probs, dim=0).numpy()
    all_labels = torch.cat(all_labels, dim=0).numpy()
    return all_probs, all_labels

In [11]:
from data_process_v4 import get_loaders
from data_process_v4 import TASK_3_TEST_LABELS_DIR
loaders = get_loaders(
    image_size=(650, 400),                 
    num_workers=0,
    test_csv_path=TASK_3_TEST_LABELS_DIR,
)

train_loader_res, val_loader_res, test_loader_res, train_df_res, val_df_res, test_df_res = loaders

loaders = get_loaders(
    image_size=(384, 384),                 
    num_workers=0,
    test_csv_path=TASK_3_TEST_LABELS_DIR,
)
train_loader_vis, val_loader_vis, test_loader_vis, train_df_vis, val_df_vis, test_df_vis = loaders

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score


probs1_val, y_val = get_probs_and_labels(best_resnet_model, val_loader_res, device)
probs2_val, _     = get_probs_and_labels(best_vis_model, val_loader_vis, device)

probs1_test, y_test = get_probs_and_labels(best_resnet_model, test_loader_res, device)
probs2_test, _     = get_probs_and_labels(best_vis_model, test_loader_vis, device)


alphas = np.linspace(0.0, 1.0, 21)  

best_alpha = None
best_recall = -1.0

for alpha in alphas:
    probs_val_stack = alpha * probs1_val + (1.0 - alpha) * probs2_val
    y_val_pred = probs_val_stack.argmax(axis=1)

    macro_recall = recall_score(y_val, y_val_pred, average="macro")
    # print(alpha, macro_recall)  # optional debugging

    if macro_recall > best_recall:
        best_recall = macro_recall
        best_alpha = alpha

print("Best alpha (val):", best_alpha)
print("Best macro recall on val:", best_recall)


Best alpha (val): 0.8
Best macro recall on val: 0.8152242937086143


In [27]:
from sklearn.metrics import classification_report, confusion_matrix
best_alpha = 0.7
# Apply best_alpha on TEST set
probs_test_stack = best_alpha * probs1_test + (1.0 - best_alpha) * probs2_test
y_test_pred = probs_test_stack.argmax(axis=1)

macro_recall_test = recall_score(y_test, y_test_pred, average="macro")
print("Macro recall (stacked, TEST):", macro_recall_test)

print(classification_report(
    y_test,
    y_test_pred,
    target_names=class_cols_order
))

cm = confusion_matrix(y_test, y_test_pred)
print("Confusion matrix:\n", cm)

Macro recall (stacked, TEST): 0.6988453111348106
              precision    recall  f1-score   support

         MEL       0.26      0.79      0.39       171
          NV       1.00      0.39      0.56       909
         BCC       0.51      0.74      0.61        93
       AKIEC       0.29      0.88      0.44        43
         BKL       0.45      0.55      0.49       217
          DF       0.60      0.68      0.64        44
        VASC       0.54      0.86      0.66        35

    accuracy                           0.51      1512
   macro avg       0.52      0.70      0.54      1512
weighted avg       0.76      0.51      0.54      1512

Confusion matrix:
 [[135   0   5  13  15   1   2]
 [327 354  37  38 123  11  19]
 [  6   0  69  12   4   1   1]
 [  1   0   2  38   2   0   0]
 [ 50   0  15  23 119   6   4]
 [  3   0   4   5   2  30   0]
 [  1   0   2   0   1   1  30]]
